In [1]:
from cloudvolume import CloudVolume
import fastremap
from tqdm.auto import tqdm
import pathlib
import json
from joblib import Parallel, delayed

/Users/mateljk1/Documents/projects/aplbrain/zebrafish-pencilsdown/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SVOX_CLOUDPATH = "precomputed://s3://cvdb-bossdb-boss/smith2024/zebrafish/supervoxels"
CKPT_CLOUDPATH_STEM = "precomputed://s3://cvdb-bossdb-boss/smith2024/zebrafish/agglomeration_checkpoint_"
COMPONENT_PATH = pathlib.Path("./component-graphs/")

RANDOMIZE_UPLOAD_ORDER = True

In [3]:
src = CloudVolume(SVOX_CLOUDPATH, progress=False)

In [4]:
zslices_at_a_time = src.chunk_size[-1]
assert zslices_at_a_time > 1

In [5]:
from random import shuffle
from intern.utils.parallel import block_compute


def int_keys_values(dct):
    return {int(k): int(v) for k, v in dct.items()}


for component_remap_file in COMPONENT_PATH.glob("*.remap.json"):
    remap_dict = json.loads(
        component_remap_file.read_text(), object_hook=int_keys_values
    )
    editcount = int(str(component_remap_file).split(".")[1])
    # Avoid rerunning ones that are already done
    if editcount in [5000, 10000, 35000, 25000, 15000, 20000, 30000, 35000, 40000]:
        continue
    # print(component_remap_file)

    cloudpath = CKPT_CLOUDPATH_STEM + str(editcount)
    dst = CloudVolume(cloudpath, info=src.info, progress=False)
    print(dst.cloudpath)
    dst.commit_info()

    def process_slice(z, zslices_at_a_time, src, dst, remap_dict):
        z2 = min(z + zslices_at_a_time, src.bounds.maxpt[2])
        src_data = src[:, :, z:z2]
        remapped_data = fastremap.remap(
            src_data, remap_dict, preserve_missing_labels=True
        )
        dst[:, :, z:z2] = remapped_data

    def process_block(block, src, dst, remap_dict):
        (x1, x2), (y1, y2), (z1, z2) = block
        src_data = src[x1:x2, y1:y2, z1:z2]
        remapped_data = fastremap.remap(
            src_data, remap_dict, preserve_missing_labels=True
        )
        dst[x1:x2, y1:y2, z1:z2] = remapped_data

    # slices = list(range(0, src.bounds.maxpt[2], zslices_at_a_time))
    # if RANDOMIZE_UPLOAD_ORDER:
    #     shuffle(slices)
    # Parallel(n_jobs=2)(
    #     delayed(process_slice)(z, zslices_at_a_time, src, dst, remap_dict)
    #     for z in tqdm(slices)
    # )
    blocks = block_compute(
        src.bounds.minpt[0],
        src.bounds.maxpt[0],
        src.bounds.minpt[1],
        src.bounds.maxpt[1],
        src.bounds.minpt[2],
        src.bounds.maxpt[2],
        block_size=(256, 256, 256),
    )
    if RANDOMIZE_UPLOAD_ORDER:
        shuffle(blocks)
    Parallel(n_jobs=16)(
        delayed(process_block)(block, src, dst, remap_dict) for block in tqdm(blocks)
    )


s3://cvdb-bossdb-boss/smith2024/zebrafish/agglomeration_checkpoint_45000


  0%|          | 32/56160 [00:10<6:11:50,  2.52it/s]/Users/mateljk1/Documents/projects/aplbrain/zebrafish-pencilsdown/.venv/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
100%|██████████| 56160/56160 [2:09:32<00:00,  7.23it/s]  


In [6]:
!say "All done"